In [3]:
# 셀 1: 경로 설정 + 파일 수정 자동 반영
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # backend/

In [2]:
# 셀 2
from agent import config
print("키 있음:", config.ALPHAVANTAGE_API_KEY is not None)
print("캐시 위치:", config.CACHE_DIR)

키 있음: True
캐시 위치: /Users/mugyeom/Desktop/개인 프로젝트/주식/backend/.cache


In [3]:
# 셀 3
from agent import mcp_client

print("등록된 서버:", list(mcp_client.client.connections.keys()))

tools = await mcp_client.get_tools("alphavantage")
print("툴 개수:", len(tools))

for t in tools:
    if t.name in ("COMPANY_OVERVIEW", "EARNINGS", "TIME_SERIES_DAILY", "TIME_SERIES_WEEKLY"):
        print(t.name, "→", t.args_schema)

등록된 서버: ['alphavantage']
툴 개수: 133
TIME_SERIES_DAILY → {'type': 'object', 'properties': {'symbol': {'type': 'string', 'description': 'The name of the equity. For example: symbol=IBM'}, 'outputsize': {'type': 'string', 'description': '"compact" (100 data points) or "full" (20+ years of historical data)'}, 'datatype': {'type': 'string', 'description': 'By default, datatype=csv. Strings json and csv are accepted with the following specifications:'}, 'entitlement': {'type': 'string', 'description': '"delayed" for 15-minute delayed data, "realtime" for realtime data'}, 'return_full_data': {'type': 'boolean', 'description': 'Parameter return_full_data'}}, 'required': ['symbol'], 'additionalProperties': False}
TIME_SERIES_WEEKLY → {'type': 'object', 'properties': {'symbol': {'type': 'string', 'description': 'The name of the equity. For example: symbol=IBM'}, 'datatype': {'type': 'string', 'description': 'By default, datatype=csv. Strings json and csv are accepted with the following specificat

In [4]:
# 셀 4
from datetime import date
from agent.adapters.base import fetch_quant_snapshot

snap = await fetch_quant_snapshot("AAPL", "US", date.today())
snap

QuantSnapshot(ticker='AAPL', market='US', as_of_date=datetime.date(2026, 10, 21), currency='USD', sector=None, industry=None, market_cap=None, price=339.5, change_pct=-1.0031, volume=35626552, per=38.9335, forward_per=None, pbr=None, roe=None, roa=None, profit_margin=None, dividend_yield=None, ma_50=326.9164, ma_200=295.9432, volatility_60d=24.8347, return_6m=25.6337, return_12m=34.5674, analyst_target_price=None, eps_estimate=1.88, eps_actual=2.02, eps_surprise_pct=7.4468)

In [5]:
# 셀 5
import time
t0 = time.time()
snap2 = await fetch_quant_snapshot("AAPL", "US", date.today())
print(f"{time.time() - t0:.3f}초, 같은 결과: {snap == snap2}")

for p in sorted(config.CACHE_DIR.rglob("*.json")):
    print(p.relative_to(config.CACHE_DIR))

0.001초, 같은 결과: True
alphavantage/2026-10-21/COMPANY_OVERVIEW_AAPL.json
alphavantage/2026-10-21/EARNINGS_AAPL.json
alphavantage/2026-10-21/TIME_SERIES_DAILY_AAPL.json
alphavantage/2026-10-21/TIME_SERIES_WEEKLY_AAPL.json


In [6]:
# 셀 6
from agent.graph import build_graph
import agent.nodes.past_decision as pd

graph = build_graph()
inp = {"ticker": "AAPL", "market": "US", "as_of_date": date.today()}

for has_past in (False, True):
    pd.HAS_PAST = has_past
    result = await graph.ainvoke(inp)       # ← invoke가 아니라 ainvoke
    print(f"\n[HAS_PAST={has_past}]")
    print("quant_summary:", result["quant_summary"])
    print("PER in report:", result["report"].quant_snapshot.per)
    print("과거 비교:", result["report"].past_decision_review is not None)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-7g0k1***************************************R3S4. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [ ]:
from agent import mcp_client

async with mcp_client.session("alphavantage") as s:
    res = await mcp_client.call_json(s, "COMPANY_OVERVIEW", {"symbol": "AAPL"})

print(list(res.keys())[:15])
print(res.get("error"))

# 리포트작성

In [4]:
from agent import config
print("OpenAI 키 있음:", config.OPENAI_API_KEY is not None, "| 모델:", config.OPENAI_MODEL)

OpenAI 키 있음: True | 모델: gpt-4o-mini


In [5]:
from datetime import date
from agent.graph import build_graph
import agent.nodes.past_decision as pd

graph = build_graph()
pd.HAS_PAST = False
result = await graph.ainvoke({"ticker": "AAPL", "market": "US", "as_of_date": date.today()})

r = result["report"]
for field in ("investment_thesis", "risks", "consensus_gap", "thesis_breakers"):
    print(f"\n[{field}]\n{getattr(r, field)}")
print("\n출처:", [s.title for s in r.sources])


[investment_thesis]
AAPL은 강력한 브랜드 파워와 지속적인 혁신을 통해 시장에서의 입지를 확고히 하고 있습니다. 최근 6개월과 12개월 수익률이 각각 25.63%와 34.57%에 달하며, 이는 투자자들의 신뢰를 반영합니다. 직전 분기 EPS가 예상치를 7.45% 초과한 것은 회사의 실적이 시장 기대를 초과했음을 보여줍니다. 이러한 요소들은 AAPL의 성장 가능성을 뒷받침하며, 높은 PER에도 불구하고 긍정적인 투자 논리를 형성합니다.

[risks]
AAPL이 직면한 주요 리스크는 높은 밸류에이션과 변동성입니다. 현재 PER이 38.93배로 상대적으로 높은 수준이며, 60일 변동성이 24.83%에 달해 주가의 급격한 변동이 우려됩니다. 또한, 회사가 밝힌 리스크가 실적에 부정적인 영향을 미칠 경우, 주가에 부정적인 영향을 줄 수 있습니다. 이러한 리스크들은 투자자에게 주의가 필요함을 시사합니다.

[consensus_gap]
현재 AAPL의 주가는 339.50 USD로, 시장의 기대치와 실제 실적 간의 간극이 존재할 수 있습니다. 직전 분기의 예상 EPS가 1.88이었던 반면, 실제 EPS는 2.02로 서프라이즈가 발생했습니다. 그러나 PER이 높은 수준인 만큼, 주가가 이미 상당 부분 기대를 반영하고 있을 가능성이 있습니다. 향후 실적이 기대에 미치지 못할 경우, 주가 조정이 발생할 수 있습니다.

[thesis_breakers]
AAPL의 투자 논리가 틀렸다고 판단할 조건은 다음과 같습니다. 첫째, 다음 분기 실제 EPS가 예상 EPS를 5% 이상 하회하는 경우. 둘째, 주가가 40주 이동평균인 295.94 USD를 10% 이상 하회하는 경우. 셋째, 60일 변동성이 30%를 초과하여 급격한 시장 불안정을 나타내는 경우.

출처: ['Alpha Vantage (EARNINGS, TIME_SERIES_DAILY, TIME_SERIES_WEEKLY)']


In [10]:
import os
from dotenv import dotenv_values
from agent import config

env_file = dotenv_values(config.ROOT_DIR / ".env").get("OPENAI_API_KEY") or ""
in_use = os.environ.get("OPENAI_API_KEY") or ""

print(".env 파일의 키  끝 4자리:", env_file[-4:], "| 길이:", len(env_file))
print("실제 사용 중인 키 끝 4자리:", in_use[-4:], "| 길이:", len(in_use))

.env 파일의 키  끝 4자리: R3S4 | 길이: 51
실제 사용 중인 키 끝 4자리: R3S4 | 길이: 51
